In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
!unzip -qo /content/drive/MyDrive/reuse_metricgan_finetune.zip -d /content/
%cd /content/reuse_metricgan_finetune
!ls

from huggingface_hub import snapshot_download
snapshot_download(repo_id="nvidia/RE-USE", local_dir="./REUSE", local_dir_use_symlinks=False)
print("RE-USE downloaded.")

if not os.path.exists('./vox-profile-release/src'):
    !git clone https://github.com/tiantiaf0627/vox-profile-release.git ./vox-profile-release
snapshot_download(repo_id="tiantiaf/whisper-large-v3-voice-quality", local_dir="./vox-profile-release", local_dir_use_symlinks=False)
print("Voice quality model downloaded.")


In [ ]:
import torch, sys, re, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

!pip install -q huggingface_hub safetensors soundfile pyyaml tqdm loralib gdown librosa
!pip install -q transformers==4.40.0

torch_ver = re.match(r'(\d+\.\d+)', torch.__version__).group(1)
py_ver = f"cp{''.join(str(x) for x in sys.version_info[:2])}"
cxx_abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
print(f"Detected: torch{torch_ver}, {py_ver}, cxx11abi{cxx_abi}")

def try_install_wheel(urls, fallback_pip):
    for url in urls:
        print(f"  Trying: {url.split('/')[-1]}")
        result = subprocess.run([sys.executable, "-m", "pip", "install", url], capture_output=True, text=True)
        if result.returncode == 0:
            print("  OK!")
            return True
        print("  Failed")
    print("  Building from source...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + fallback_pip)
    return True

causal_urls = [
    f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1%2Bcu12torch{torch_ver}cxx11abi{cxx_abi}-{py_ver}-{py_ver}-linux_x86_64.whl"
]
mamba_urls = [
    f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1%2Bcu12torch{torch_ver}cxx11abi{cxx_abi}-{py_ver}-{py_ver}-linux_x86_64.whl"
]
print("\nInstalling causal-conv1d:")
try_install_wheel(causal_urls, ["causal-conv1d>=1.4.0", "--no-build-isolation"])
print("\nInstalling mamba-ssm:")
try_install_wheel(mamba_urls, ["mamba-ssm", "--no-build-isolation"])

print("\n" + "="*60)
print("  ALL INSTALLED.")
print("  NOW: Runtime -> Restart runtime")
print("  Then run Cell 3 onward. Do NOT re-run Cells 1-2.")
print("="*60)


In [ ]:
import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")
print(f"vRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import transformers
print(f"transformers: {transformers.__version__}")
assert transformers.__version__.startswith("4.40"), f"Wrong transformers! Got {transformers.__version__}"

import mamba_ssm
print(f"mamba-ssm: {mamba_ssm.__version__}")
print("\nEnvironment OK!")


In [ ]:
import os, gdown

%cd /content/reuse_metricgan_finetune

os.makedirs("./data/urgent2025", exist_ok=True)

print("Downloading URGENT 2025 official validation set (noisy)...")
gdown.download(id="1Ip-C5tUNGCssT8KAjHUUoh99jkzRH6nm", output="./data/urgent2025/noisy_val.zip", quiet=False)
print("Downloading URGENT 2025 official validation set (clean)...")
gdown.download(id="11geBBf24WKN1xT_NasnI4JrmKpqNo8h9", output="./data/urgent2025/clean_val.zip", quiet=False)

print("Downloading URGENT 2025 non-blind test set (noisy)...")
gdown.download(id="1rxV6RgA4LAp2I1EnHsln7wI7-UCP6Qer", output="./data/urgent2025/noisy_test.zip", quiet=False)
print("Downloading URGENT 2025 non-blind test set (clean)...")
gdown.download(id="1RarjxOgWkaDV8EjH_eLX169y89PVa3sg", output="./data/urgent2025/clean_test.zip", quiet=False)

print("\nUnzipping...")
!unzip -qo ./data/urgent2025/noisy_val.zip -d ./data/urgent2025/train_noisy_raw/
!unzip -qo ./data/urgent2025/clean_val.zip -d ./data/urgent2025/train_clean_raw/
!unzip -qo ./data/urgent2025/noisy_test.zip -d ./data/urgent2025/test_noisy_raw/
!unzip -qo ./data/urgent2025/clean_test.zip -d ./data/urgent2025/test_clean_raw/

print("\nContents:")
!find ./data/urgent2025/train_noisy_raw -type f -name '*.flac' | head -5
!echo "---"
!find ./data/urgent2025/train_clean_raw -type f -name '*.flac' | head -5


In [ ]:
import os, shutil, glob

def flatten_audio(src_root, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    count = 0
    for ext in ['*.flac', '*.wav']:
        for fpath in glob.glob(os.path.join(src_root, '**', ext), recursive=True):
            fname = os.path.basename(fpath)
            dst = os.path.join(dst_dir, fname)
            if not os.path.exists(dst):
                shutil.copy2(fpath, dst)
                count += 1
    return count

print("Flattening training data...")
n1 = flatten_audio('./data/urgent2025/train_noisy_raw', './data/urgent2025/train/noisy')
n2 = flatten_audio('./data/urgent2025/train_clean_raw', './data/urgent2025/train/clean')
print(f"  Train: {n1} noisy, {n2} clean")

print("Flattening test data...")
n3 = flatten_audio('./data/urgent2025/test_noisy_raw', './data/urgent2025/test/noisy')
n4 = flatten_audio('./data/urgent2025/test_clean_raw', './data/urgent2025/test/clean')
print(f"  Test: {n3} noisy, {n4} clean")


train_noisy = set(os.path.splitext(f)[0] for f in os.listdir('./data/urgent2025/train/noisy'))
train_clean = set(os.path.splitext(f)[0] for f in os.listdir('./data/urgent2025/train/clean'))
train_pairs = train_noisy & train_clean

test_noisy = set(os.path.splitext(f)[0] for f in os.listdir('./data/urgent2025/test/noisy'))
test_clean = set(os.path.splitext(f)[0] for f in os.listdir('./data/urgent2025/test/clean'))
test_pairs = test_noisy & test_clean

print(f"\nMatched pairs: {len(train_pairs)} train, {len(test_pairs)} test")
print("Data ready!")


In [ ]:
import sys, os, time, json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import soundfile as sf
import librosa
from huggingface_hub import hf_hub_download

device = torch.device('cuda')
%cd /content/reuse_metricgan_finetune

sys.path.insert(0, os.path.abspath('./REUSE'))
sys.path.insert(0, os.path.abspath('./vox-profile-release'))

from models.stfts import mag_phase_stft, mag_phase_istft
from models.generator_SEMamba_time_d4 import SEMamba
from utils.util import load_config, pad_or_trim_to_match

config_path = hf_hub_download(repo_id='nvidia/RE-USE', filename='config.json')
cfg = load_config(config_path)
SE_model = SEMamba.from_pretrained('nvidia/RE-USE', cfg=cfg).to(device)
SE_model.train()
print(f"RE-USE loaded ({sum(p.numel() for p in SE_model.parameters())/1e6:.1f}M params)")

from discriminator import MetricDiscriminator
D = MetricDiscriminator(n_fft=512, hop_length=128).to(device)
print(f"Discriminator loaded ({sum(p.numel() for p in D.parameters())/1e3:.1f}K params)")

from vq_metric import VOICE_QUALITY_LABELS, NUM_LABELS
from src.model.voice_quality.whisper_voice_quality import WhisperWrapper

vq_model = WhisperWrapper.from_pretrained("tiantiaf/whisper-large-v3-voice-quality").float().to(device)
vq_model.eval()
for p in vq_model.parameters():
    p.requires_grad_(False)
print(f"Voice quality model loaded (frozen). {NUM_LABELS} labels.")

print(f"\nGPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GB")


In [ ]:
import random
from torch.utils.data import Dataset, DataLoader


TRAIN_NOISY_DIR = './data/urgent2025/train/noisy'
TRAIN_CLEAN_DIR = './data/urgent2025/train/clean'
TEST_NOISY_DIR = './data/urgent2025/test/noisy'
TEST_CLEAN_DIR = './data/urgent2025/test/clean'
CHECKPOINT_DIR = '/content/drive/MyDrive/metricgan_checkpoints'
SAMPLE_RATE = 16000
SEGMENT_LENGTH = 4 * SAMPLE_RATE
BATCH_SIZE = 4
NUM_EPOCHS = 10
LR_G = 1e-5
LR_D = 1e-4
LAMBDA_SPEC = 1.0
LAMBDA_GAN = 0.5
VQ_EVAL_EVERY = 50
SAVE_EVERY = 200
LOG_EVERY = 10


os.makedirs(CHECKPOINT_DIR, exist_ok=True)


class UrgentDataset(Dataset):
    def __init__(self, noisy_dir, clean_dir, segment_length, sample_rate=16000):
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.segment_length = segment_length
        self.sample_rate = sample_rate

        noisy_stems = {os.path.splitext(f)[0]: f for f in os.listdir(noisy_dir)
                       if f.endswith(('.flac', '.wav'))}
        clean_stems = {os.path.splitext(f)[0]: f for f in os.listdir(clean_dir)
                       if f.endswith(('.flac', '.wav'))}
        common = sorted(set(noisy_stems) & set(clean_stems))
        self.pairs = [(noisy_stems[s], clean_stems[s]) for s in common]
        print(f"  Dataset: {len(self.pairs)} pairs from {noisy_dir}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        noisy_f, clean_f = self.pairs[idx]
        noisy, sr_n = sf.read(os.path.join(self.noisy_dir, noisy_f), dtype='float32')
        clean, sr_c = sf.read(os.path.join(self.clean_dir, clean_f), dtype='float32')

        if noisy.ndim > 1: noisy = noisy.mean(axis=-1)
        if clean.ndim > 1: clean = clean.mean(axis=-1)

        if sr_n != self.sample_rate:
            noisy = librosa.resample(noisy, orig_sr=sr_n, target_sr=self.sample_rate)
        if sr_c != self.sample_rate:
            clean = librosa.resample(clean, orig_sr=sr_c, target_sr=self.sample_rate)

        min_len = min(len(noisy), len(clean))
        noisy, clean = noisy[:min_len], clean[:min_len]

        if min_len >= self.segment_length:
            start = random.randint(0, min_len - self.segment_length)
            noisy = noisy[start:start + self.segment_length]
            clean = clean[start:start + self.segment_length]
        else:
            noisy = np.pad(noisy, (0, self.segment_length - min_len))
            clean = np.pad(clean, (0, self.segment_length - min_len))

        return torch.from_numpy(noisy).float(), torch.from_numpy(clean).float()

print("Loading datasets...")
train_dataset = UrgentDataset(TRAIN_NOISY_DIR, TRAIN_CLEAN_DIR, SEGMENT_LENGTH)
test_dataset = UrgentDataset(TEST_NOISY_DIR, TEST_CLEAN_DIR, SEGMENT_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=1)

print(f"Train: {len(train_dataset)} pairs, {len(train_loader)} batches/epoch")
print(f"Test: {len(test_dataset)} pairs")


RELU = nn.ReLU()
n_fft_cfg = cfg['stft_cfg']['n_fft']
hop_size_cfg = cfg['stft_cfg']['hop_size']
win_size_cfg = cfg['stft_cfg']['win_size']
compress_factor = cfg['model_cfg']['compress_factor']
sampling_rate_cfg = cfg['stft_cfg']['sampling_rate']

def make_even(v):
    v = int(round(v))
    return v if v % 2 == 0 else v + 1

def enhance(wav_batch):
    sr = SAMPLE_RATE
    n_fft_s = make_even(n_fft_cfg * sr // sampling_rate_cfg)
    hop_s = make_even(hop_size_cfg * sr // sampling_rate_cfg)
    win_s = make_even(win_size_cfg * sr // sampling_rate_cfg)
    noisy_mag, noisy_pha, _ = mag_phase_stft(wav_batch, n_fft=n_fft_s, hop_size=hop_s,
                                              win_size=win_s, compress_factor=compress_factor,
                                              center=True, addeps=False)
    amp_g, pha_g, _ = SE_model(noisy_mag, noisy_pha)
    mag = torch.expm1(RELU(amp_g))
    zero_portion = torch.sum(mag == 0, 1) / mag.shape[1]
    amp_g[:, :, (zero_portion > 0.5)[0]] = 0
    audio_g = mag_phase_istft(amp_g, pha_g, n_fft_s, hop_s, win_s, compress_factor)
    audio_g = pad_or_trim_to_match(wav_batch.detach(), audio_g, pad_value=1e-8)
    return audio_g

@torch.no_grad()
def compute_vq_distance(enhanced_wav, clean_wav):
    def get_probs(wav):
        if wav.shape[0] < SAMPLE_RATE:
            return None
        data = wav[:15*SAMPLE_RATE].unsqueeze(0).float().to(device)
        logits = vq_model(data, return_feature=False)
        return torch.sigmoid(logits).cpu().numpy()[0]
    p1 = get_probs(enhanced_wav.cpu())
    p2 = get_probs(clean_wav.cpu())
    if p1 is None or p2 is None:
        return 0.5
    return float(np.abs(p1 - p2).sum() / NUM_LABELS)


class STFTLoss(nn.Module):
    def __init__(self, n_fft=512, hop=128):
        super().__init__()
        self.n_fft = n_fft
        self.hop = hop
    def forward(self, x, y):
        window = torch.hann_window(self.n_fft, device=x.device)
        x_spec = torch.stft(x, self.n_fft, self.hop, self.n_fft, window=window, return_complex=True).abs()
        y_spec = torch.stft(y, self.n_fft, self.hop, self.n_fft, window=window, return_complex=True).abs()
        sc = torch.norm(y_spec - x_spec, p='fro') / (torch.norm(y_spec, p='fro') + 1e-8)
        mag = nn.functional.l1_loss(torch.log(x_spec + 1e-8), torch.log(y_spec + 1e-8))
        return sc + mag

class MultiResSTFTLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.losses = nn.ModuleList([
            STFTLoss(512, 128),
            STFTLoss(1024, 256),
            STFTLoss(256, 64),
        ])
    def forward(self, x, y):
        return sum(l(x, y) for l in self.losses) / len(self.losses)

mr_stft_loss = MultiResSTFTLoss().to(device)
l1_loss_fn = nn.L1Loss()


opt_g = optim.AdamW(SE_model.parameters(), lr=LR_G, weight_decay=0.01)
opt_d = optim.AdamW(D.parameters(), lr=LR_D, weight_decay=0.01)

scheduler_g = optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=NUM_EPOCHS * len(train_loader))
scheduler_d = optim.lr_scheduler.CosineAnnealingLR(opt_d, T_max=NUM_EPOCHS * len(train_loader))


history = {'step': [], 'loss_g': [], 'loss_spec': [], 'loss_gan': [], 'loss_d': [], 'vq_dist': []}
global_step = 0
best_vq_dist = float('inf')

print(f"\n{'='*60}")
print(f"  Starting training: {NUM_EPOCHS} epochs, {len(train_loader)} steps/epoch")
print(f"  Total steps: ~{NUM_EPOCHS * len(train_loader)}")
print(f"  Batch size: {BATCH_SIZE}, LR_G: {LR_G}, LR_D: {LR_D}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"{'='*60}\n")

t_start = time.time()

for epoch in range(NUM_EPOCHS):
    SE_model.train()
    D.train()
    epoch_losses = {'g': [], 'spec': [], 'gan': [], 'd': []}

    for batch_idx, (noisy_batch, clean_batch) in enumerate(train_loader):
        noisy_batch = noisy_batch.to(device)
        clean_batch = clean_batch.to(device)

        enhanced = enhance(noisy_batch)

        loss_l1 = l1_loss_fn(enhanced, clean_batch)
        loss_stft = mr_stft_loss(enhanced, clean_batch)
        loss_spec = loss_l1 + loss_stft

        if global_step % VQ_EVAL_EVERY == 0:
            vq_dists = torch.tensor([
                compute_vq_distance(enhanced[j].detach(), clean_batch[j])
                for j in range(BATCH_SIZE)
            ], device=device)

            opt_d.zero_grad()
            pred_d = D(enhanced.detach(), clean_batch).squeeze(-1)
            loss_d = nn.functional.mse_loss(pred_d, vq_dists)
            loss_d.backward()
            opt_d.step()
            scheduler_d.step()

            last_vq_dist = vq_dists.mean().item()
        else:
            loss_d = torch.tensor(0.0)
            scheduler_d.step()

        opt_g.zero_grad()
        pred_g = D(enhanced, clean_batch).squeeze(-1)
        loss_gan = nn.functional.mse_loss(pred_g, torch.zeros_like(pred_g))
        loss_g = LAMBDA_SPEC * loss_spec + LAMBDA_GAN * loss_gan
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(SE_model.parameters(), 5.0)
        opt_g.step()
        scheduler_g.step()

        epoch_losses['g'].append(loss_g.item())
        epoch_losses['spec'].append(loss_spec.item())
        epoch_losses['gan'].append(loss_gan.item())
        epoch_losses['d'].append(loss_d.item())

        if global_step % LOG_EVERY == 0:
            elapsed = time.time() - t_start
            vq_str = f"VQ={last_vq_dist:.4f}" if 'last_vq_dist' in dir() else ""
            print(f"  [{epoch+1}/{NUM_EPOCHS}] Step {global_step:4d} | "
                  f"L_g={loss_g.item():.4f} L_spec={loss_spec.item():.4f} "
                  f"L_gan={loss_gan.item():.4f} L_d={loss_d.item():.4f} "
                  f"{vq_str} | {elapsed:.0f}s")

        history['step'].append(global_step)
        history['loss_g'].append(loss_g.item())
        history['loss_spec'].append(loss_spec.item())
        history['loss_gan'].append(loss_gan.item())
        history['loss_d'].append(loss_d.item())
        history['vq_dist'].append(last_vq_dist if 'last_vq_dist' in dir() else 0)

        if global_step > 0 and global_step % SAVE_EVERY == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f'checkpoint_step{global_step}.pt')
            torch.save({
                'step': global_step,
                'epoch': epoch,
                'generator': SE_model.state_dict(),
                'discriminator': D.state_dict(),
                'opt_g': opt_g.state_dict(),
                'opt_d': opt_d.state_dict(),
                'history': history,
            }, ckpt_path)
            print(f"    >> Checkpoint saved: {ckpt_path}")

        global_step += 1

    print(f"\n  Epoch {epoch+1} done | "
          f"avg L_g={np.mean(epoch_losses['g']):.4f} "
          f"avg L_spec={np.mean(epoch_losses['spec']):.4f} "
          f"avg L_gan={np.mean(epoch_losses['gan']):.4f}\n")


final_path = os.path.join(CHECKPOINT_DIR, 'checkpoint_final.pt')
torch.save({
    'step': global_step,
    'epoch': NUM_EPOCHS,
    'generator': SE_model.state_dict(),
    'discriminator': D.state_dict(),
    'opt_g': opt_g.state_dict(),
    'opt_d': opt_d.state_dict(),
    'history': history,
}, final_path)
print(f"\nFinal checkpoint saved: {final_path}")
print(f"Total training time: {(time.time()-t_start)/60:.1f} minutes")


In [ ]:
print("Evaluating fine-tuned model on test set...")
SE_model.eval()
finetuned_dists = []
n_eval = min(100, len(test_dataset))

with torch.no_grad():
    for i in range(n_eval):
        noisy, clean = test_dataset[i]
        noisy_gpu = noisy.unsqueeze(0).to(device)
        enhanced = enhance(noisy_gpu)
        dist = compute_vq_distance(enhanced[0], clean.to(device))
        finetuned_dists.append(dist)
        if (i+1) % 20 == 0:
            print(f"  {i+1}/{n_eval} evaluated...")


print("\nEvaluating baseline (pretrained RE-USE)...")
SE_baseline = SEMamba.from_pretrained('nvidia/RE-USE', cfg=cfg).to(device)
SE_baseline.eval()
baseline_dists = []

def enhance_baseline(wav_batch):
    sr = SAMPLE_RATE
    n_fft_s = make_even(n_fft_cfg * sr // sampling_rate_cfg)
    hop_s = make_even(hop_size_cfg * sr // sampling_rate_cfg)
    win_s = make_even(win_size_cfg * sr // sampling_rate_cfg)
    noisy_mag, noisy_pha, _ = mag_phase_stft(wav_batch, n_fft=n_fft_s, hop_size=hop_s,
                                              win_size=win_s, compress_factor=compress_factor,
                                              center=True, addeps=False)
    amp_g, pha_g, _ = SE_baseline(noisy_mag, noisy_pha)
    mag = torch.expm1(RELU(amp_g))
    zero_portion = torch.sum(mag == 0, 1) / mag.shape[1]
    amp_g[:, :, (zero_portion > 0.5)[0]] = 0
    audio_g = mag_phase_istft(amp_g, pha_g, n_fft_s, hop_s, win_s, compress_factor)
    audio_g = pad_or_trim_to_match(wav_batch.detach(), audio_g, pad_value=1e-8)
    return audio_g

with torch.no_grad():
    for i in range(n_eval):
        noisy, clean = test_dataset[i]
        noisy_gpu = noisy.unsqueeze(0).to(device)
        enhanced = enhance_baseline(noisy_gpu)
        dist = compute_vq_distance(enhanced[0], clean.to(device))
        baseline_dists.append(dist)
        if (i+1) % 20 == 0:
            print(f"  {i+1}/{n_eval} evaluated...")

del SE_baseline
torch.cuda.empty_cache()


print(f"\n{'='*60}")
print(f"  EVALUATION RESULTS ({n_eval} test samples)")
print(f"{'='*60}")
print(f"  Baseline RE-USE VQ distance:    {np.mean(baseline_dists):.4f} +/- {np.std(baseline_dists):.4f}")
print(f"  Fine-tuned RE-USE VQ distance:  {np.mean(finetuned_dists):.4f} +/- {np.std(finetuned_dists):.4f}")
improvement = np.mean(baseline_dists) - np.mean(finetuned_dists)
print(f"  Improvement:                    {improvement:+.4f} ({'BETTER' if improvement > 0 else 'worse'})")
print(f"  Relative improvement:           {improvement/np.mean(baseline_dists)*100:+.1f}%")
print(f"{'='*60}")


results = {
    'baseline_mean_vq_dist': float(np.mean(baseline_dists)),
    'finetuned_mean_vq_dist': float(np.mean(finetuned_dists)),
    'improvement': float(improvement),
    'relative_improvement_pct': float(improvement/np.mean(baseline_dists)*100),
    'n_eval_samples': n_eval,
    'total_training_steps': global_step,
    'epochs': NUM_EPOCHS,
    'final_spec_loss': float(history['loss_spec'][-1]),
}
with open('/content/drive/MyDrive/metricgan_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved to Drive: metricgan_results.json")
